# Structure and Modularization of the Code

## Principles for Production Apps

To transform prototypes into robust production apps the code organization becomes a critical point. Usually for someone that comes from the data world this might seem odd but for us that joined tech through "default programming" this is something that is already part of our daily job. So we might see similar structures as the ones we already use in production for commom applications.

| Principle | Description | Benefits |
| --- | --- | --- |
| Separation of Concerns | Isolate components by responsibilities | - Simplified support - Reusability (DRY) - Testability |
| Documentation | Document the intention and behaviour | - Faster Onboarding - Colaboration - Faster support and maintainability |
| Auto verification | Implement analysis tools (linters, CI/CD, TDD, etc) | - Bug prevention - Code consistency - issue detection |

## Recommended Structure
    projeto_pydanticai/,
    ├── app/,
    │   ├── agents/           # Agents definition,
    │   ├── models/           # Pydantic models for input and outputs,
    │   ├── prompts/          # System prompts and templates,
    │   ├── tools/            # Tools impl,
    │   └── utils/            # General Aux functions,
    ├── config/               # Environment Configurations,
    ├── tests/                # Automated Tests,
    └── main.py               # Entrypoint,

# Quality Practices

#### Type Checking and Validation

In [ ]:
# Recommended Configuration for pyproject.toml

[tool.mypy]
python_version = "3.13"
disallow_untyped_defs = true
disallow_incomplete_defs = true

[tool.ruff]
target-version = "py313"
line-length = 88
select = ["E", "F", "B", "I"]

#### Component Documentation

In [ ]:
class ProductRecommendationAgent:
    """
    Specialized agent for product recommendation.
    
    This agent uses the user profile and purchase history, using a RAG strategy to find the most relevant products.
    
    Attributes:
        product_db: Database connection with product information.
        user_history: List of previous purchases.
    """

> Tip: Create an internal lib for reusing components (prompts, tools, models) that can be shared between different agents to minimize duplication.

## Automated Tests

Create unit tests for each component of the aget.

For example, test every single tool individualy, the validation of the model and the execution flow.

In [ ]:
from dataclasses import dataclass
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext

load_dotenv()

class FakeOrderSystem:
    def get_order_status(self, order_id: str) -> str:
        return "In Progress" if order_id == "123" else "Completed"
    
@dataclass
class TestDeps:
    order_system: FakeOrderSystem
    
agent = Agent(
    'openrouter:google/gemini-2.5-flash',
    deps_type=TestDeps,
    system_prompt="You are an assistant that helps users search their orders"
)

@agent.tool()
async def get_order_status(ctx: RunContext[TestDeps], order_id: str) -> str:
    """
    Search the status of an order in the system.
    
    Args: 
        ctx: The context with access to the order system
        order_id: ID of the order to be searched
        
    Returns:
        The status of the order
    """
    return ctx.deps.order_system.get_order_status(order_id)

async def test_through_agent():
    fake_deps = TestDeps(
        order_system=FakeOrderSystem()
    )
    
    valid_result = await agent.run(
        "What is the status of the order 123?",
        deps=fake_deps
    )
    
    invalid_result = await agent.run(
        "What is the status of the order 456?",
        deps=fake_deps
    )
    print("Answer for the valid order:")
    print(valid_result.output)
    print("\nAnswer for the invalid order:")
    print(invalid_result.output)
    
await test_through_agent()

123
456
Answer for the valid order:
The order 123 is In Progress.

Answer for the invalid order:
The order 456 is completed.


## Analysis of the Agent test

1. The tool get_order_status is working with dependency injection (making easier for us to test).
2. The agent is integrating the tool correctly - Can call the tool with the correct data.
3. The dependency injection flow is correct - The ovject FakerOrderSystem is being injected and used.

## What we've learned

1. Integration tests vs. Unit tests - When making tests for Agents sometimes is better to test end to end the integration
2. Mocks of dependencies - The class FakeOrderSystem shows how we can simulate the real tool without making a call to a database for example
3. Execution Context - The RunContext should not be created manually- let the PydanticAI manage it.
4. Result analysis - LLMs can vary the exact shape of the response, so it's better to check if the main information are present instead of a 1:1 match.

## Monitoring Logs in Production

### Tracking agent performance

AI systems in production require proactive monitoring to ensure the functional correctness and the confiability of the system

| Component | Objective | Implementation |
| --- | --- | --- |
| Structured Logs | Detailed tracing of the app | Pydantic Logfire (We can use OTel) |
| Latency metrics | Monitor TTFT or other metrics | Capture the timestamp per node in the graph |
| Token Usage | Cost control | Automatic tracing per request |
| Tool tracing | usage analysis | Logs of calls and response of tools |


In [ ]:
# Ex of configuration of logfire
from pydantic_ai.logfire import setup_logfire_monitoring

setup_logfire_monitoring(
    level="INFO",
    capture_stats=True,
    log_directory="./logs",
    rotation="10 MB"
)

> Performance Tip: Configure alerts for latency and errors anomalies to identify errors before aftecting real users